In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

df_marketing_raw = spark.table(
    "lh_bronze_game.marketing_spend_raw"
)

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 3, Finished, Available, Finished, False)

In [2]:
print("Row count:", df_marketing_raw.count())
print("Column count:", len(df_marketing_raw.columns))

df_marketing_raw.printSchema()

display(df_marketing_raw.limit(5))

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 4, Finished, Available, Finished, False)

Row count: 1973
Column count: 12
root
 |-- date: date (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- country: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- creative_id: string (nullable = true)
 |-- creative_type: string (nullable = true)
 |-- creative_concept: string (nullable = true)
 |-- campaign_objective: string (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- installs: integer (nullable = true)
 |-- spend_usd: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 310e235b-aa39-469d-9c36-73ba5ad2197f)

In [3]:
df_marketing_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_marketing_raw.columns
]).show(truncate=False)

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 5, Finished, Available, Finished, False)

+----+-------------------+-------+-----------+-----------+-------------+----------------+------------------+-----------+------+--------+---------+
|date|acquisition_channel|country|campaign_id|creative_id|creative_type|creative_concept|campaign_objective|impressions|clicks|installs|spend_usd|
+----+-------------------+-------+-----------+-----------+-------------+----------------+------------------+-----------+------+--------+---------+
|0   |0                  |0      |0          |0          |0            |0               |0                 |0          |0     |0       |0        |
+----+-------------------+-------+-----------+-----------+-------------+----------------+------------------+-----------+------+--------+---------+



In [4]:
print(
    "Exact duplicate rows:",
    df_marketing_raw.count() - df_marketing_raw.dropDuplicates().count()
)

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 6, Finished, Available, Finished, False)

Exact duplicate rows: 0


In [5]:
df_marketing_raw.select(
    F.count(
        F.when(F.col("impressions") < 0, 1)
    ).alias("negative_impressions"),

    F.count(
        F.when(F.col("clicks") < 0, 1)
    ).alias("negative_clicks"),

    F.count(
        F.when(F.col("installs") < 0, 1)
    ).alias("negative_installs"),

    F.count(
        F.when(F.col("spend_usd") < 0, 1)
    ).alias("negative_spend"),

    F.count(
        F.when(F.col("clicks") > F.col("impressions"), 1)
    ).alias("clicks_gt_impressions"),

    F.count(
        F.when(F.col("installs") > F.col("clicks"), 1)
    ).alias("installs_gt_clicks")
).show()

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 7, Finished, Available, Finished, False)

+--------------------+---------------+-----------------+--------------+---------------------+------------------+
|negative_impressions|negative_clicks|negative_installs|negative_spend|clicks_gt_impressions|installs_gt_clicks|
+--------------------+---------------+-----------------+--------------+---------------------+------------------+
|                   0|              0|                0|             0|                    0|                 0|
+--------------------+---------------+-----------------+--------------+---------------------+------------------+



In [6]:
df_marketing_clean = df_marketing_raw

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 8, Finished, Available, Finished, False)

In [7]:
df_marketing_clean.groupBy("acquisition_channel").count().show()
df_marketing_clean.groupBy("creative_type").count().show()
df_marketing_clean.groupBy("campaign_objective").count().show()

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 9, Finished, Available, Finished, False)

+-------------------+-----+
|acquisition_channel|count|
+-------------------+-----+
|          Unity Ads|  538|
|         TikTok Ads|  476|
|           Meta Ads|  334|
|         Google Ads|  625|
+-------------------+-----+

+-------------+-----+
|creative_type|count|
+-------------+-----+
|Fail Scenario|  324|
|     Gameplay|  494|
|    Character|  384|
|  Progression|  352|
|       Reward|  262|
|    UGC Style|  157|
+-------------+-----+

+------------------+-----+
|campaign_objective|count|
+------------------+-----+
| Payer Acquisition|  502|
|       Retargeting|  431|
|     Re-engagement|  192|
|           Install|  848|
+------------------+-----+



In [8]:
orphan_campaigns = (
    df_marketing_clean
    .select("campaign_id")
    .distinct()
    .alias("m")
    .join(
        spark.table("lh_silver_game.players_clean")
            .filter(F.col("campaign_id").isNotNull())
            .select("campaign_id")
            .distinct()
            .alias("p"),
        on="campaign_id",
        how="left_anti"
    )
)

print("Players tablosunda hiç görünmeyen campaign_id:",
      orphan_campaigns.count())

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 10, Finished, Available, Finished, False)

Players tablosunda hiç görünmeyen campaign_id: 0


In [9]:
orphan_creatives = (
    df_marketing_clean
    .select("creative_id")
    .distinct()
    .alias("m")
    .join(
        spark.table("lh_silver_game.players_clean")
            .filter(F.col("creative_id").isNotNull())
            .select("creative_id")
            .distinct()
            .alias("p"),
        on="creative_id",
        how="left_anti"
    )
)

print("Players tablosunda hiç görünmeyen creative_id:",
      orphan_creatives.count())

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 11, Finished, Available, Finished, False)

Players tablosunda hiç görünmeyen creative_id: 0


In [10]:
df_marketing_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_silver_game.marketing_spend_clean")

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 12, Finished, Available, Finished, False)

In [11]:
df_check = spark.table("lh_silver_game.marketing_spend_clean")

print("Saved row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, c21c1815-d844-4783-bf4c-771aa4a5cafc, 13, Finished, Available, Finished, False)

Saved row count: 1973


SynapseWidget(Synapse.DataFrame, ec5afb11-0c33-425d-ae7b-e89bb40fa85b)